Q1

In [45]:
import numpy as np

Q1 -  Reading a Reaction SMILES

In [46]:
reactions = [
    "CC(=O)O.CCO>[H+].[Cl-]>CC(=O)OCC.O",
    "C=C.[H][H]>[Pd]>CC",
    "c1ccccc1.O=[N+]([O-])O>>c1ccccc1[N+](=O)[O-].O"
]

def parse_reaction(rxn):
    rxn_splt = rxn.split(">")
    if len(rxn_splt) != 3:
        raise ValueError("reaction must contain exactly two > separators")
    d = {}
    for k, s in zip(["reactants", "reagents", "products"], rxn_splt):
        d[k] = [] if s == "" else s.split(".")
    return d

for r in reactions:
    d = parse_reaction(r)
    for k in d:
        print(k, len(d[k]), d[k])
    print()

reactants 2 ['CC(=O)O', 'CCO']
reagents 2 ['[H+]', '[Cl-]']
products 2 ['CC(=O)OCC', 'O']

reactants 2 ['C=C', '[H][H]']
reagents 1 ['[Pd]']
products 1 ['CC']

reactants 2 ['c1ccccc1', 'O=[N+]([O-])O']
reagents 0 []
products 2 ['c1ccccc1[N+](=O)[O-]', 'O']



Q2 - Balancing Ethane Combustion as a null Space Problem

In [47]:
E = np.array([[2, 0, -1, 0],
              [6, 0,  0, -2],
              [0, 2, -2, -1]], float)

u, s, vt = np.linalg.svd(E)
x = vt[-1]
x = x / x[0]
m = 2
c = np.rint(x * m).astype(int)

print("Singular Values:", s)
print("The Rank:", np.linalg.matrix_rank(E), "nullity of E:", E.shape[1] - np.linalg.matrix_rank(E))
print("x:", x)
print("Coefficients:", c)
print("E @ x:", E @ x)

Singular Values: [6.62561256 3.00720721 1.02857332]
The Rank: 3 nullity of E: 1
x: [1.  3.5 2.  3. ]
Coefficients: [2 7 4 6]
E @ x: [-4.44089210e-16 -1.77635684e-15 -8.88178420e-16]


# Q3 - Huckel Spectrum of butadiene


In [48]:
A4 = np.zeros((4,4))
for i in range(0,4):
  if i+1 < 4 :
    A4[i][i+1] = 1
    A4[i+1][i] = 1
print("Adjacency Matrix A4 (Butadiene Chain):\n", A4)


degrees = np.sum(A4, axis=1)
print("\nDegree of every atom in A4:", degrees)


eigenvalues = np.linalg.eigvalsh(A4)
print("\nFour eigenvalues of A4 (sorted ascending):", eigenvalues)


E_pi_coeff = 2 * (eigenvalues[0] + eigenvalues[1])
print(f"\nE_pi (total pi-electron energy) = {E_pi_coeff:.4f} * beta")
delocalization_energy_coeff = E_pi_coeff - 4
print(f"Delocalization Energy = {delocalization_energy_coeff:.4f} * beta")

print(f"\nButadiene's delocalization energy ({delocalization_energy_coeff:.4f} * beta) is less than benzene's (2 * beta), indicating butadiene is less stabilized by delocalization compared to the highly aromatic benzene ring.")

Adjacency Matrix A4 (Butadiene Chain):
 [[0. 1. 0. 0.]
 [1. 0. 1. 0.]
 [0. 1. 0. 1.]
 [0. 0. 1. 0.]]

Degree of every atom in A4: [1. 2. 2. 1.]

Four eigenvalues of A4 (sorted ascending): [-1.61803399 -0.61803399  0.61803399  1.61803399]

E_pi (total pi-electron energy) = -4.4721 * beta
Delocalization Energy = -8.4721 * beta

Butadiene's delocalization energy (-8.4721 * beta) is less than benzene's (2 * beta), indicating butadiene is less stabilized by delocalization compared to the highly aromatic benzene ring.


Q4 - Oversmoothing of Benzene Ring

In [49]:
A = np.zeros((6, 6))
for i in range(6):
    A[i, (i + 1) % 6] = A[(i + 1) % 6, i] = 1.0

At = A + np.eye(6)
P = At / At.sum(axis=1, keepdims=True)

H = np.random.default_rng(1).normal(size=(6, 3))

v = np.linalg.eigvals(P)
v = np.sort(np.abs(v))[::-1]
mu = v[1]

print("eigenvalue moduli:", v)
print("mu:", mu)

for k in [0, 1, 2, 4, 8, 16]:
    Z = H.copy()
    for _ in range(k):
        Z = P @ Z
    d = np.max(np.linalg.norm(Z - Z.mean(axis=0), axis=1))
    print(k, d)

eigenvalue moduli: [1.00000000e+00 6.66666667e-01 6.66666667e-01 3.33333333e-01
 7.91848035e-17 3.92523115e-17]
mu: 0.6666666666666667
0 1.2413880520727198
1 0.5368253372377662
2 0.3459101833698203
4 0.1548726398897235
8 0.030667573455258054
16 0.0011967999497354295


Interpretation: The deviation shrinks quickly as layers increase, so for a molecule with about a dozen heavy atoms only a small number of message-passing layers is useful before node features become too similar.

Q5 - PCA of Two Descriptors

In [50]:
X = np.array([[78.1, 2.3],
              [92.1, 1.7],
              [106.2, 2.8],
              [120.2, 0.7],
              [134.2, 3.1],
              [148.2, 2.4]])

def pca(X):
    Xp = X - X.mean(axis=0)
    C = Xp.T @ Xp / len(X)
    w, v = np.linalg.eigh(C)
    j = np.argsort(w)[::-1]
    w, v = w[j], v[:, j]
    for i in range(v.shape[1]):
        q = np.argmax(np.abs(v[:, i]))
        if v[q, i] < 0:
            v[:, i] *= -1
    return C, w, w / w.sum(), v

C, w, f, v = pca(X)
Z = (X - X.mean(axis=0)) / X.std(axis=0)
Cz, wz, fz, vz = pca(Z)

print("Centered covariance:\n", C)
print("eigenvalues:", w)
print("fraction of variance:", f)
print("Centered PC1:", v[:, 0])

print("\nstandardized covariance:\n", Cz)
print("eigenvalues:", wz)
print("fraction of variance:", fz)
print("Standardised PC1:", vz[:, 0])

Centered covariance:
 [[573.53555556   3.03888889]
 [  3.03888889   0.61888889]]
eigenvalues: [573.55167411   0.60277034]
fraction of variance: [0.99895016 0.00104984]
Centered PC1: [0.99998593 0.00530402]

standardized covariance:
 [[1.         0.16129775]
 [0.16129775 1.        ]]
eigenvalues: [1.16129775 0.83870225]
fraction of variance: [0.58064887 0.41935113]
Standardised PC1: [0.70710678 0.70710678]


Interpretation: The centered PCA is dominated by molar mass because its numerical scale is much larger, whereas standardized PCA gives both descriptors equal scale, so the standardized result is the better choice when both descriptors should contribute fairly.